## **Maths for LoRA (Low-Rank Adaptation)**

### **Rank of a Matrix**

The rank of a matrix is defined as the maximum number of `linearly independent` rows or columns in the matrix. It provides insight into the dimensionality of the vector space spanned by its rows or columns.

For example, consider a matrix $A$:

$$
A = \begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$

The rank of matrix $A$ can be determined by performing row reduction or calculating the determinant. In this case, the rank of $A$ is 2, as there are two linearly independent rows (or columns).

Because in the matrix $A$, the third row can be expressed as a linear combination of the first two rows:

$$
\text{Row}_3 = \text{Row}_1 + 2 \cdot \text{Row}_2
$$  



<hr>
<hr>

## **LoRA (Low-Rank Adaptation)**

### 1. The Origin Story: Why LoRA Was Invented

In late 2020 and early 2021, the natural language processing landscape underwent a massive transition. OpenAI had recently published their work on GPT-3 (175 Billion parameters). Researchers across the industry realized a fundamental bottleneck: while these massive models had unprecedented zero-shot capabilities, fine-tuning them for highly specific downstream tasks (such as specialized legal analysis, medical extraction, or structural code generation) was becoming financially and logistically impossible for almost everyone.

Fine-tuning GPT-3 (175B) required:
* Over **1.2 Terabytes** of GPU memory just to store the trainable parameters, gradients, and optimizer states.
* Tens of thousands of dollars in compute costs per training run.
* Storing distinct 700 GB checkpoints for every customized task.

In June 2021, a team at Microsoft Research (Edward J. Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, and Weizhu Chen) published a paper titled **"LoRA: Low-Rank Adaptation of Large Language Models"**. Their goal was to build an adaptation framework that required almost no VRAM overhead, added **zero** latency during inference, and matched or exceeded the performance of Full Fine-Tuning (FFT).

---

### 2. The Intrinsic Dimension Principle: The Foundation of LoRA

To understand why LoRA works, you must understand the mathematical theory it relies on: **The Intrinsic Dimension of Objective Landscapes**.

#### The Theory
In 2018, Li et al. published a landmark paper showing that deep, over-parameterized neural networks actually operate in a much lower-dimensional subspace than their parameter count suggests. 

In 2020, Aghajanyan et al. applied this concept specifically to pre-trained language models. They discovered that:
1. When a model is pre-trained on massive, diverse datasets, it compresses the fundamental structures of language into its weights.
2. When you adapt this model to a specific task (like sentiment analysis), you do not need to search the entire, high-dimensional parameter space ($D$).
3. The parameter updates ($\Delta W$) required to adapt the model to a new task actually reside on a subspace with a much lower **"intrinsic dimension"** ($d \ll D$).

#### The LoRA Hypothesis
The authors of LoRA generalized this finding to matrix ranks. They hypothesized that:
> The change in weights ($\Delta W$) during model adaptation has a **low "intrinsic rank."**

Mathematically, if you have a pre-trained weight projection matrix $W_0 \in \mathbb{R}^{d \times k}$ (where $d$ and $k$ are hidden dimensions, e.g., $4096$), its mathematical maximum possible rank is $\min(d, k) = 4096$. 

During fine-tuning, the updates are captured by $\Delta W \in \mathbb{R}^{d \times k}$. The LoRA hypothesis states that because the update is highly directed and narrow (dedicated to only the downstream task), $\Delta W$ is fundamentally **rank-deficient**. Its effective, necessary rank $r$ is extremely small (e.g., $r = 4$ or $r = 8$) compared to the full dimension of $4096$.

---

### 3. Did Adapters "Fail"? The Shift to LoRA

Before LoRA, sequential bottleneck adapters (like Houlsby adapters) were the state-of-the-art PEFT method. They did not mathematically "fail"—they still successfully reduced parameter counts and matched full fine-tuning performance. However, they fell out of favor for large-scale, high-throughput production due to structural issues:

#### Why Adapters Are Impractical in Modern LLM Serving
1. **The Sequential Latency Bottleneck:** Adapters place a sequential down-projection, non-linear activation ($f(\cdot)$), and up-projection step directly inside the active Transformer pipeline. 
   Because of this non-linear function $f(\cdot)$, **you cannot merge adapters back into the original weights.** 
   At inference time, the model must run these extra layers sequentially for every single generated token. Across 32+ Transformer layers, this adds noticeable latency, which is highly problematic for real-time applications.
2. **GPU Memory Bandwidth Saturation:** Modern LLM inference is highly memory-bandwidth bound (the time is spent moving weights from GPU memory to the GPU cache, rather than performing math). Adding extra distinct sequential steps forces the GPU to repeatedly write and read intermediate activations from memory, reducing hardware efficiency.
3. **Graph Modification Complexity:** Adapters require altering the internal forward pass graph of standard Transformer blocks. This complicates tensor-parallel slicing across multiple GPUs and breaks integration with optimized kernel libraries like FlashAttention.

#### Why LoRA Won
LoRA solved all of these problems by running **in parallel** to the existing linear layers during training, and **merging completely** into the base weights during deployment. This leaves the model's native architecture completely unchanged, resulting in **zero extra latency** at inference time.

---

### 4. Mathematical Foundations of LoRA

Let's break down the exact mathematics of how LoRA factorizes a matrix update.

#### Low-Rank Matrix Decomposition
Suppose we want to update a frozen linear layer weight $W_0 \in \mathbb{R}^{d \times k}$. The update matrix is $\Delta W \in \mathbb{R}^{d \times k}$. 
If we constrain $\Delta W$ to have a maximum rank of $r$ (where $r \ll \min(d, k)$), we can factorize $\Delta W$ into the product of two thin matrices:

$$\Delta W = B \times A$$

Where:
* $A \in \mathbb{R}^{r \times k}$ (Input dimension to rank dimension)
* $B \in \mathbb{R}^{d \times r}$ (Rank dimension to output dimension)

```bash
                 Original Weight Matrix W0               Update Matrix ΔW
                      (d x k)                                (d x k)
                 ┌───────────────────┐                 ┌───────────────────┐
                 │                   │                 │                   │
                 │                   │                 │                   │
                 │                   │                 │                   │
                 └───────────────────┘                 └───────────────────┘
                                                                 │
                                                    Decomposed into:
                                                                 ▼
                                                    Matrix B           Matrix A
                                                    (d x r)            (r x k)
                                                     ┌───┐
                                                     │   │  x  ┌───────────────────┐
                                                     │   │     │                   │
                                                     └───┘     └───────────────────┘
```

#### The Scaling Factor $\frac{\alpha}{r}$
The complete LoRA forward pass equation is:

$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (BA) x$$

Where:
* $\alpha$ is a constant scaling hyperparameter (often set to 16 or 32).
* $r$ is the chosen rank (e.g., 8).

##### Why use the $\frac{\alpha}{r}$ multiplier?
When we alter the rank $r$, the magnitude of the adapter weights scales. If we do not scale the outputs, changing the rank hyperparameter would force us to completely re-tune our learning rate and weight decay hyperparameters.

By scaling the output by $\frac{\alpha}{r}$, the contribution of the adapter pathway is normalized. If you decide to change the rank from $r=8$ to $r=16$, the magnitude of the gradient updates remains mathematically stable, allowing you to reuse the same learning rate. Typically, $\alpha$ is kept constant as a multiple of your starting rank (e.g., if you choose $r=8$, you set $\alpha=16$).

#### Initialization Mechanics: Why $B$ is Zero
At the start of training:
* Matrix $A$ is initialized using a random Gaussian distribution: $A \sim \mathcal{N}\left(0, \sigma^2\right)$.
* Matrix $B$ is initialized to **exactly zero**: $B = 0$.

Let's look at the mathematical consequence of this initialization at Step 0:
$$\Delta W = B \times A = 0 \times A = 0$$

$$\text{Forward Pass at Step 0}: \quad h = W_0 x + \frac{\alpha}{r} (0 \cdot A) x = W_0 x$$

This initialization is critical because it ensures that **at the start of training, the model behaves exactly like the pre-trained base model.** There is no random noise introduced to the outputs, which stabilizes early training and eliminates the need for long warm-up phases.

---

### 5. Step-by-Step Forward and Backward Passes in LoRA

Let's trace the detailed forward and backward mathematical passes for a single linear projection layer inside a Transformer (e.g., the Query projection matrix $W_q$).

```bash
                      Input Vector: x
                      ┌─────────────┐
                      │             │
                      └──────┬──────┘
             ┌───────────────┴───────────────┐
             ▼                               ▼
     Frozen Base Path               Trainable LoRA Path
    ┌─────────────────┐             ┌─────────────────┐
    │   Matrix W0     │             │    Matrix A     │
    │    (d x k)      │             │    (r x k)      │
    └────────┬────────┘             └────────┬────────┘
             │                               ▼
             │                       Intermediate: z_mid
             │                      ┌─────────────────┐
             │                      │    Matrix B     │
             │                      │    (d x r)      │
             │                      └────────┬────────┘
             ▼                               ▼
          h_base                          h_lora
             │                               │
             └───────────────┬───────────────┘
                             ▼
                    Combine: h = h_base + (alpha/r) * h_lora
```

#### The Forward Pass
1. Input $x \in \mathbb{R}^{N \times k}$ is received.
2. **Base path:** Compute the frozen projection:
   $$h_{\text{base}} = x W_0^T \quad \left[ N \times d \right]$$
3. **LoRA path step 1:** Project the input down to the bottleneck rank $r$:
   $$z = x A^T \quad \left[ N \times r \right]$$
4. **LoRA path step 2:** Project back up to the hidden dimension $d$:
   $$h_{\text{lora}} = z B^T \quad \left[ N \times d \right]$$
5. **Scale and Merge:** Combine both pathways:
   $$h = h_{\text{base}} + \frac{\alpha}{r} h_{\text{lora}} = x W_0^T + \frac{\alpha}{r} \left( x A^T B^T \right)$$

---

#### The Backward Pass (Gradients)
During backpropagation, we receive the gradient of the loss with respect to the output layer: $\frac{\partial \mathcal{L}}{\partial h} = G \in \mathbb{R}^{N \times d}$. 

We must calculate the gradients for our active parameters ($A$ and $B$) as well as the gradient to propagate back to the input ($x$).

#### 1. Gradient with respect to Matrix $B$
Using the chain rule, we compute the gradient for $B$ (recalling that $h_{\text{lora}} = z B^T \implies \frac{\partial h}{\partial B} = \frac{\alpha}{r} z$):

$$\frac{\partial \mathcal{L}}{\partial B} = \frac{\alpha}{r} G^T z \quad \left[ (d \times N) \times (N \times r) \implies d \times r \right]$$

#### 2. Gradient with respect to the Intermediate State $z$
To calculate the gradient for matrix $A$, we first propagate the gradient back through the linear layer of $B$:

$$\frac{\partial \mathcal{L}}{\partial z} = \frac{\alpha}{r} G B \quad \left[ (N \times d) \times (d \times r) \implies N \times r \right]$$

#### 3. Gradient with respect to Matrix $A$
Now we use $\frac{\partial \mathcal{L}}{\partial z}$ to compute the gradient for $A$ (since $z = x A^T$):

$$\frac{\partial \mathcal{L}}{\partial A} = \left( \frac{\partial \mathcal{L}}{\partial z} \right)^T x \quad \left[ (r \times N) \times (N \times k) \implies r \times k \right]$$

#### 4. Gradient with respect to the Input $x$
Finally, we propagate the gradient back to the input vector $x$ so preceding layers can continue their calculations. Note that since $W_0$ is frozen, we do not compute $\frac{\partial \mathcal{L}}{\partial W_0}$, but the gradient must still pass through $W_0$ back to $x$:

$$\frac{\partial \mathcal{L}}{\partial x} = G W_0 + \left( \frac{\partial \mathcal{L}}{\partial z} \right) A \quad \left[ (N \times d) \times (d \times k) + (N \times r) \times (r \times k) \implies N \times k \right]$$

*The base weight matrix $W_0$ is only used here as a static multiplier. No optimizer tracking or weight updates occur for it, preserving VRAM.*

---

### 6. Real-World Architecture Example (e.g., LLaMA-3 Block)

To see where LoRA is applied in modern architectures, let's look at a single Transformer block of a modern LLM like **LLaMA-3-8B** ($d_{\text{model}} = 4096$).

#### Where do we place LoRA?
In early LoRA implementations, adapters were only applied to the Query ($W_q$) and Value ($W_v$) projection layers inside the attention mechanism. However, subsequent research (such as the QLoRA paper) demonstrated that applying LoRA to **all linear layers** yields significantly better performance, matching or exceeding Full Fine-Tuning even at very low ranks ($r = 4$ or $r = 8$).

Inside a modern LLaMA-3 Transformer Layer, LoRA adapters are typically attached to:
1. **Attention Blocks:** $W_q, W_k, W_v, W_o$
2. **MLP/Feed-Forward Blocks:** $W_{\text{gate}}, W_{\text{up}}, W_{\text{down}}$

```bash
Standard Transformer Block with LoRA Integration:

[Input: x] ────────────────────┬──────────────────────────────────────────┐
                               ▼                                          │
                         [LayerNorm]                                      │
                               ▼                                          │
                  ┌──────────────────────────┐                            │
                  │   Multi-Head Attention   │                            │
                  │  (Targeted with LoRA)    │                            │
                  │  - Wq + (Bq x Aq)        │                            │
                  │  - Wk + (Bk x Ak)        │                            │
                  │  - Wv + (Bv x Av)        │                            │
                  │  - Wo + (Bo x Ao)        │                            │
                  └────────────┬─────────────┘                            │
                               ▼                                          │
                         [LayerNorm]                                      │
                               ▼                                          │
                  ┌──────────────────────────┐                            │
                  │   Feed-Forward Network   │                            │
                  │  (Targeted with LoRA)    │                            │
                  │  - Wgate + (Bgate x Agate)                            │
                  │  - Wup   + (Bup   x Aup)                              │
                  │  - Wdown + (Bdown x Adown)                            │
                  └────────────┬─────────────┘                            │
                               ▼                                          │
                             ( + ) <──────────────────────────────────────┘
                               │
                               ▼
                        [Output: y]
```

#### Parameter Trade-offs across Different Ranks ($r$)
Let's calculate the exact parameter impact on a single LLaMA-3 layer projection ($W_q \in \mathbb{R}^{4096 \times 4096}$):

* **No LoRA (Full Fine-Tuning):** $4096 \times 4096 = \mathbf{16,777,216 \text{ parameters}}$

* **LoRA with Rank $r = 4$:**
  * Matrix $A \in \mathbb{R}^{4 \times 4096} = 16,384$
  * Matrix $B \in \mathbb{R}^{4096 \times 4} = 16,384$
  * **Trainable Parameters:** $16,384 + 16,384 = \mathbf{32,768}$ *(99.8% reduction)*

* **LoRA with Rank $r = 8$:**
  * Matrix $A \in \mathbb{R}^{8 \times 4096} = 32,768$
  * Matrix $B \in \mathbb{R}^{4096 \times 8} = 32,768$
  * **Trainable Parameters:** $32,768 + 32,768 = \mathbf{65,536}$ *(99.6% reduction)*

* **LoRA with Rank $r = 16$:**
  * Matrix $A \in \mathbb{R}^{16 \times 4096} = 65,536$
  * Matrix $B \in \mathbb{R}^{4096 \times 16} = 65,536$
  * **Trainable Parameters:** $65,536 + 65,536 = \mathbf{131,072}$ *(99.2% reduction)*

Even with a higher rank of $r=16$, the trainable parameter footprint remains a tiny fraction of the original projection matrix. This reduction in trainable parameters is what enables rapid, memory-efficient fine-tuning on consumer-grade hardware.